In [1]:
import os
from dotenv import load_dotenv
from datasets import Dataset
from pandas import DataFrame

# Updated Ragas metric imports
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas import evaluate

# LangChain OpenAI integrations
from ragas.llms import LangchainLLMWrapper
# Use Groq for the Judge and HuggingFace for fast, local embeddings
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
# 1. Load the environment variables from your .env file
load_dotenv()

# Verify the key is loaded (don't print the actual key in output!)
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY not found. Please check your .env file.")

# 2. Initialize Models
# Requires GROQ_API_KEY in your .env file
judge_model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print(f"✅ Judge LLM configured as: {judge_model.model_name}")
print(f"✅ Embeddings configured as: {embeddings.model_name}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Judge LLM configured as: llama-3.1-8b-instant
✅ Embeddings configured as: sentence-transformers/all-MiniLM-L6-v2


In [3]:
data = {
    'question': [
        'What is Kubernetes?',
        'What is Docker?'
    ],
    'answer': [
        'Kubernetes is an open-source system for automating deployment, scaling, and management of containerized applications.',
        # 'Docker is a platform designed to help developers build, share, and run modern applications.'
        'A Docker is a device that helps ship dock at the port.'
    ],
    'contexts': [
        ['Kubernetes is an open-source container orchestration system.', 'Docker is used to create and run containers either on bare-metal or cloud run virtualized infra on top of hypervisors.'],
        ['Kubernetes is an open-source container orchestration system.', 'Docker is used to create and run containers either on bare-metal or cloud run virtualized infra on top of hypervisors.']
    ],
    'reference': [
        'Kubernetes is an open-source system for managing containerized applications.',
        'Docker is a platform for building and running containers.'
    ]
}

dataset = Dataset.from_dict(data)
display(DataFrame(dataset))

,question,answer,contexts,reference
0,What is Kubernetes?,Kubernetes is an open-source system for automa...,[Kubernetes is an open-source container orches...,Kubernetes is an open-source system for managi...
1,What is Docker?,A Docker is a device that helps ship dock at t...,[Kubernetes is an open-source container orches...,Docker is a platform for building and running ...


In [4]:
print("Evaluating model outputs concurrently via Groq / HuggingFace...")

# Execute Ragas using the cloud models
score = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=judge_model,
    embeddings=embeddings,
    raise_exceptions=False # Prevents the entire run from failing if one evaluation errors out
)

# Output as a clean Pandas DataFrame for analysis
df_results = score.to_pandas()
display(DataFrame(df_results))

Evaluating model outputs concurrently via Groq / HuggingFace...


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]


--- Evaluation Results ---
{'faithfulness': 0.5000, 'answer_relevancy': 0.9467, 'context_precision': 0.7500, 'context_recall': 1.0000}


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,What is Kubernetes?,[Kubernetes is an open-source container orches...,Kubernetes is an open-source system for automa...,Kubernetes is an open-source system for managi...,1.0,0.919421,1.0,1.0
1,What is Docker?,[Kubernetes is an open-source container orches...,A Docker is a device that helps ship dock at t...,Docker is a platform for building and running ...,0.0,0.974016,0.5,1.0


# Understanding the RAGAS Scores

Consider the second example:

## Input

### Question
```text
What is Docker?
```

### Retrieved Context
```text
Docker is used to create and run containers either on bare-metal or cloud run virtualized infra on top of hypervisors.
```

### Reference
```text
Docker is a platform for building and running containers.
```

### Generated Answer
```text
A Docker is a device that helps ship dock at the port.
```

---

# Why is Context Recall High?

At first glance, this seems surprising.

The generated answer talks about ships and ports, while the retrieved context talks about containers and containerization.

One might expect Context Recall to be very low.

However, **Context Recall is not evaluating the generated answer.**

Instead, it evaluates whether the retrieved context contains the information needed to answer the question according to the reference answer.

Conceptually:

```text
Context Recall ≈ f(Question, Context, Reference)
```

and **not**

```text
Context Recall ≈ f(Question, Context, Answer)
```

In this example:

### Reference
```text
Docker is a platform for building and running containers.
```

### Retrieved Context
```text
Docker is used to create and run containers...
```

These two statements are semantically very similar:

```text
building/running containers
≈
creating/running containers
```

Therefore, the retriever successfully returned the information needed to answer the question.

As a result:

```text
Context Recall ≈ 1.0
```

even though the generated answer is completely wrong.

---

# Why is Context Precision High?

Context Precision measures whether the retrieved context is relevant to the question.

Question:

```text
What is Docker?
```

Retrieved Context:

```text
Docker is used to create and run containers...
```

The context is highly relevant to the question and contains useful information about Docker.

Therefore:

```text
Context Precision ≈ High
```

The generated answer does not directly affect this metric.

---

# Why is Faithfulness Low?

Faithfulness evaluates whether the generated answer is supported by the retrieved context.

Retrieved Context:

```text
Docker is used to create and run containers...
```

Generated Answer:

```text
A Docker is a device that helps ship dock at the port.
```

The answer introduces information that is completely absent from the context.

There is no evidence in the context supporting:

```text
device
ships
ports
docking
```

Therefore:

```text
Faithfulness ≈ 0
```

This is exactly the behavior we want.

---

# Why is Response Relevancy High?

This is another metric that often surprises people.

Response Relevancy does **not** ask:

```text
Is the answer correct?
```

Instead, it asks something closer to:

```text
Is the answer attempting to address the question?
```

Question:

```text
What is Docker?
```

Answer:

```text
A Docker is a device that helps ship dock at the port.
```

The answer is clearly attempting to define "Docker."

The answer is wrong, but it is still on-topic.

Compare:

### Wrong but Relevant
```text
What is Docker?

Docker is a device that helps ship dock at the port.
```

### Irrelevant
```text
What is Docker?

Paris is the capital of France.
```

The first answer is incorrect but still related to the subject of the question.

The second answer is completely unrelated.

As a result, many LLM judges assign:

```text
Response Relevancy ≈ High
```

for the first case.

---

# Key Insight

Many newcomers implicitly expect:

```text
Wrong Answer
⇒
Low Scores Everywhere
```

However, RAGAS intentionally separates retrieval quality from generation quality.

In this example:

| Component | Result |
|------------|----------|
| Retriever | Successful |
| Retrieved Context | Relevant |
| Retrieved Context | Sufficient |
| Generator | Failed |
| Answer | Hallucinated |

This explains the observed pattern:

| Metric | Expected Result |
|----------|----------|
| Context Recall | High |
| Context Precision | High |
| Faithfulness | Low |
| Response Relevancy | High |
| Hallucination | High |

---

# Diagnostic Interpretation

A useful rule of thumb when analyzing RAGAS results:

### Pattern 1

```text
Context Recall = High
Context Precision = High
Faithfulness = Low
```

Interpretation:

```text
Retrieval succeeded.
Generation failed.
```

The system found the correct information but failed to use it properly.

---

### Pattern 2

```text
Context Recall = Low
Faithfulness = Low
```

Interpretation:

```text
Retrieval failed.
Generation never received the required information.
```

The root cause is likely in retrieval rather than generation.

---

# Takeaway

This example is a textbook case of:

```text
Successful Retrieval + Hallucinated Generation
```

The retriever returned the right information about Docker, but the generator ignored it and produced an unsupported answer.

This illustrates one of the key strengths of RAGAS: it helps identify *where* a RAG pipeline is failing rather than simply reporting whether the final answer was correct or incorrect.